# TreeSpace functionality guide

This notebook is a practical guide to the new `TreeSpace` API in SpaceCore.

`TreeSpace` replaces the old `ProductSpace` idea. It represents a finite **direct product** of leaf spaces, but stores elements using a Python tree: tuples, lists, dicts, named tuples, dataclasses registered with `optree`, or any structure supported by `optree`.

Mathematically, a `TreeSpace` is

\[
X = \prod_{\ell \in L} X_\ell.
\]

The tree controls representation only. It is **not** a tensor product.

Run this notebook from the repository root with the project `.venv` kernel.


## 0. Setup

The code below assumes that the current working directory is the SpaceCore repository root, or that SpaceCore is installed in the active `.venv`.

In [1]:
from __future__ import annotations

import pathlib
from typing import NamedTuple

import numpy as np
import optree
from optree.dataclasses import dataclass as pytree_dataclass

import spacecore as sc

repo = pathlib.Path.cwd()
print("SpaceCore version:", getattr(sc, "__version__", "unknown"))
print("Current directory:", repo)


SpaceCore version: 0.3.2
Current directory: /Users/pavlopelikh/Documents/Dev/SpaceCore/notebooks


In [2]:
ctx = sc.Context(sc.NumpyOps(), dtype=np.float64, check_level="standard")

# Small helper for display/assertions in this notebook.
def as_np(x):
    return np.asarray(x)

## 1. Basic tuple-structured TreeSpace

The simplest tree is a tuple. This is the direct replacement for the old two-component product-space use case.

Here the mathematical space is

\[
X = \mathbb{R}^2 \times \mathbb{R}^3.
\]

The element representation is a Python tuple `(x0, x1)`.

In [3]:
X0 = sc.DenseCoordinateSpace((2,), ctx)
X1 = sc.DenseCoordinateSpace((3,), ctx)

T = sc.TreeSpace.from_leaf_spaces((X0, X1), ctx)

print("TreeSpace type:", type(T).__name__)
print("arity:", T.arity)
print("dense coordinate shape:", T.shape)
print("leaf paths:", T.leaf_paths)
print("leaf spaces:", [type(s).__name__ for s in T.leaf_spaces])

TreeSpace type: TreeInnerProductSpace
arity: 2
dense coordinate shape: (5,)
leaf paths: ((0,), (1,))
leaf spaces: ['DenseCoordinateSpace', 'DenseCoordinateSpace']


In [4]:
x = (
    ctx.asarray([1.0, 2.0]),
    ctx.asarray([10.0, 20.0, 30.0]),
)

T.check(x)
print("x is a valid TreeSpace element")

x is a valid TreeSpace element


## 2. Tree flattening versus coordinate flattening

`TreeSpace` has two different flattening notions:

1. `flatten_tree(x)` returns the ordered Python leaves.
2. `flatten(x)` returns one dense coordinate vector.

The first preserves Python objects. The second is the coordinate representation used by coordinate-space methods and linear algebra.

In [5]:
leaves = T.flatten_tree(x)
flat = T.flatten(x)
roundtrip = T.unflatten(flat)

print("ordered leaves:", leaves)
print("dense vector:", flat)
print("roundtrip:", roundtrip)

assert len(leaves) == 2
assert np.allclose(as_np(flat), [1.0, 2.0, 10.0, 20.0, 30.0])
assert np.allclose(as_np(roundtrip[0]), as_np(x[0]))
assert np.allclose(as_np(roundtrip[1]), as_np(x[1]))

ordered leaves: (array([1., 2.]), array([10., 20., 30.]))
dense vector: [ 1.  2. 10. 20. 30.]
roundtrip: (array([1., 2.]), array([10., 20., 30.]))


## 3. Leafwise vector-space operations

`zero`, `ones`, `add`, and `scale` are applied leaf by leaf and then rebuilt into the same tree structure.

In [6]:
y = (
    ctx.asarray([3.0, 4.0]),
    ctx.asarray([100.0, 200.0, 300.0]),
)

z = T.zeros()
s = T.scale(2.0, x)
xy = T.add(x, y)

print("zero:", z)
print("2*x:", s)
print("x+y:", xy)

assert np.allclose(as_np(z[0]), [0.0, 0.0])
assert np.allclose(as_np(s[1]), [20.0, 40.0, 60.0])
assert np.allclose(as_np(xy[0]), [4.0, 6.0])

zero: (array([0., 0.]), array([0., 0., 0.]))
2*x: (array([2., 4.]), array([20., 40., 60.]))
x+y: (array([4., 6.]), array([110., 220., 330.]))


## 4. Arbitrary tree templates and leaf order

A `TreeSpace` is not limited to tuples. Use `from_template` when you want a named or nested structure.

The important rule is:

> `leaf_spaces` must be passed in the same order as `TreeSpace.leaf_paths`.

For dictionaries, `optree` does not use the semantic order you may have in mind. In the nested dict below, the deterministic leaf order is `dual.alpha`, `dual.beta`, then `state`.

This is why an earlier variable named `u` could fail validation: the error was not caused by the name `u`; it was caused by pairing the spaces with the wrong tree paths.


In [7]:
template = {
    "state": 0,
    "dual": {
        "alpha": 0,
        "beta": 0,
    },
}

State = sc.DenseCoordinateSpace((2,), ctx)
Alpha = sc.DenseCoordinateSpace((1,), ctx)
Beta = sc.DenseCoordinateSpace((1,), ctx)

# Inspect the leaf order before passing leaf spaces.
raw_treedef = optree.tree_structure(template)
print("optree leaf paths:")
for path in optree.treespec_paths(raw_treedef):
    print(" ", path)

# Correct order: dual.alpha, dual.beta, state.
Structured = sc.TreeSpace.from_template(template, (Alpha, Beta, State), ctx=ctx)

print("\nTreeSpace leaf paths and spaces:")
for path, leaf_space in zip(Structured.leaf_paths, Structured.leaf_spaces):
    print(" ", path, "->", leaf_space.shape)


optree leaf paths:
  ('dual', 'alpha')
  ('dual', 'beta')
  ('state',)

TreeSpace leaf paths and spaces:
  ('dual', 'alpha') -> (1,)
  ('dual', 'beta') -> (1,)
  ('state',) -> (2,)


In [8]:
u = Structured.ones()
u_half = Structured.axpy(-0.5, u, u)  # equals 0.5*u leafwise

print("u:", u)
print("0.5*u:", u_half)
print("flatten_tree(u):", Structured.flatten_tree(u))
print("flatten(u):", Structured.flatten(u))
print("unflatten(flatten(u)):", Structured.unflatten(Structured.flatten(u)))

Structured.check(u)
Structured.check(u_half)
assert np.allclose(as_np(Structured.flatten(u_half)), 0.5 * as_np(Structured.flatten(u)))


u: {'state': array([1., 1.]), 'dual': {'alpha': array([1.]), 'beta': array([1.])}}
0.5*u: {'state': array([0.5, 0.5]), 'dual': {'alpha': array([0.5]), 'beta': array([0.5])}}
flatten_tree(u): (array([1.]), array([1.]), array([1., 1.]))
flatten(u): [1. 1. 1. 1.]
unflatten(flatten(u)): {'state': array([1., 1.]), 'dual': {'alpha': array([1.]), 'beta': array([1.])}}


### Why the earlier `u` error happened

The following cell intentionally reproduces the failure mode. Here the spaces are passed as `(State, Alpha, Beta)`, but the tree paths are ordered as `dual.alpha`, `dual.beta`, `state`. Therefore `State` is assigned to `dual.alpha`, not to `state`.

A semantically natural value with a two-dimensional `state` leaf then fails validation because `state` is paired with the wrong one-dimensional leaf space.


In [9]:
WrongStructured = sc.TreeSpace.from_template(template, (State, Alpha, Beta), ctx=ctx)

semantic_u = {
    "state": ctx.asarray([1.0, 2.0]),
    "dual": {
        "alpha": ctx.asarray([0.5]),
        "beta": ctx.asarray([-0.5]),
    },
}

print("Wrong leaf pairing:")
for path, leaf_space in zip(WrongStructured.leaf_paths, WrongStructured.leaf_spaces):
    print(" ", path, "->", leaf_space.shape)

try:
    WrongStructured.check(semantic_u)
except Exception as exc:
    print("\nExpected validation error:")
    print(type(exc).__name__)
    print(exc)


Wrong leaf pairing:
  ('dual', 'alpha') -> (2,)
  ('dual', 'beta') -> (1,)
  ('state',) -> (1,)

Expected validation error:
SpaceValidationError
Invalid leaf at $.dual.alpha for DenseCoordinateSpace: Expected shape (2,), got (1,)


In [10]:
manual_u = {
    "state": ctx.asarray([1.0, 2.0]),
    "dual": {
        "alpha": ctx.asarray([0.5]),
        "beta": ctx.asarray([-0.5]),
    },
}

Structured.check(manual_u)
print("manual_u is valid for the correctly ordered TreeSpace")
print("flatten(manual_u):", Structured.flatten(manual_u))


manual_u is valid for the correctly ordered TreeSpace
flatten(manual_u): [ 0.5 -0.5  1.   2. ]


## 5. Named tuples preserve semantic structure

Named tuples are useful when an algorithm has mathematically named components and you want tuple-like behavior with attribute access.


In [11]:
class PrimalDual(NamedTuple):
    primal: object
    dual: object

PD = sc.TreeSpace.from_template(
    PrimalDual(primal=0, dual=0),
    (sc.DenseCoordinateSpace((2,), ctx), sc.DenseCoordinateSpace((1,), ctx)),
    ctx=ctx,
)

pd = PrimalDual(
    primal=ctx.asarray([1.0, 2.0]),
    dual=ctx.asarray([3.0]),
)

rebuilt = PD.unflatten(PD.flatten(pd))
print(rebuilt)
print(type(rebuilt), rebuilt.primal, rebuilt.dual)

assert isinstance(rebuilt, PrimalDual)

PrimalDual(primal=array([1., 2.]), dual=array([3.]))
<class '__main__.PrimalDual'> [1. 2.] [3.]


## 6. Dataclasses as tree structures

Dataclasses are useful when algorithm variables have stable mathematical names and you want normal attribute access.

Use `optree.dataclasses.dataclass`, not the standard-library `dataclasses.dataclass`, if you want the dataclass fields to be TreeSpace leaves. In the current TreeSpace implementation, use global optree registration with `namespace=""`, because `TreeSpace` calls `optree.tree_structure` without a namespace.

A plain standard-library dataclass is treated by `optree` as a single leaf, which is usually not what you want for TreeSpace.


In [15]:
@pytree_dataclass(namespace="")
class Variables:
    state: object
    alpha: object
    beta: object

DataclassSpace = sc.TreeSpace.from_template(
    Variables(state=0, alpha=0, beta=0),
    (State, Alpha, Beta),
    ctx=ctx,
)

d = Variables(
    state=ctx.asarray([1.0, 2.0]),
    alpha=ctx.asarray([0.25]),
    beta=ctx.asarray([-0.25]),
)

print("leaf paths:", DataclassSpace.leaf_paths)
print("flatten_tree:", DataclassSpace.flatten_tree(d))
print("flatten:", DataclassSpace.flatten(d))

rebuilt = DataclassSpace.unflatten(DataclassSpace.flatten(d))
print("rebuilt:", rebuilt)
print("rebuilt.state:", rebuilt.state)

assert isinstance(rebuilt, Variables)
assert np.allclose(as_np(rebuilt.state), [1.0, 2.0])
assert np.allclose(as_np(rebuilt.alpha), [0.25])
assert np.allclose(as_np(rebuilt.beta), [-0.25])


leaf paths: (('state',), ('alpha',), ('beta',))
flatten_tree: (array([1., 2.]), array([0.25]), array([-0.25]))
flatten: [ 1.    2.    0.25 -0.25]
rebuilt: Variables(state=array([1., 2.]), alpha=array([0.25]), beta=array([-0.25]))
rebuilt.state: [1. 2.]


In [13]:
# A TreeElement can also bind dataclass leaves explicitly.
d_element = DataclassSpace.element(d)
print("TreeElement leaves:", d_element.leaves)
print("TreeElement value:", d_element.value)

assert isinstance(d_element.value, Variables)


TreeElement leaves: (array([1., 2.]), array([0.25]), array([-0.25]))
TreeElement value: Variables(state=array([1., 2.]), alpha=array([0.25]), beta=array([-0.25]))


## 7. `TreeElement`: optional explicit binding

Most SpaceCore methods accept ordinary matching trees. `TreeElement` is useful when you want an explicit object saying “these leaves belong to this exact `TreeSpace`”.


In [ ]:
element = sc.TreeElement(T, T.flatten_tree(x))

print("TreeElement leaves:", element.leaves)
print("TreeElement value:", element.value)

assert np.allclose(as_np(T.flatten(element)), as_np(T.flatten(x)))

## 8. Validation errors identify tree structure and leaves

`TreeSpace.check` validates both the tree structure and every leaf. Leaf failures include the leaf path where possible.


In [ ]:
bad_structure = [ctx.asarray([1.0, 2.0]), ctx.asarray([3.0, 4.0, 5.0])]

try:
    T.check(bad_structure)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

In [ ]:
bad_leaf = (
    ctx.asarray([1.0, 2.0]),
    ctx.asarray([3.0, 4.0]),  # wrong shape: expected (3,)
)

try:
    T.check(bad_leaf)
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

## 9. Batches are trees of batched leaves

For a leading-axis batch, each leaf has its own leading batch dimension. `flatten_batch` concatenates leaf coordinates into an `(N, size)` array.


In [ ]:
xb = (
    ctx.asarray([[1.0, 2.0], [3.0, 4.0]]),
    ctx.asarray([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]]),
)

flat_b = T.flatten_batch(xb)
roundtrip_b = T.unflatten_batch(flat_b)

print("flat batch shape:", flat_b.shape)
print(flat_b)

assert flat_b.shape == (2, 5)
assert np.allclose(as_np(roundtrip_b[0]), as_np(xb[0]))
assert np.allclose(as_np(roundtrip_b[1]), as_np(xb[1]))

## 10. Inner product and norm

If every leaf is an inner-product space, then `TreeSpace` advertises inner-product behavior. The inner product is the sum of leaf inner products.


In [ ]:
inner_xy = T.inner(x, y)
norm_x = T.norm(x)

manual_inner = np.vdot(as_np(x[0]), as_np(y[0])) + np.vdot(as_np(x[1]), as_np(y[1]))
manual_norm = np.sqrt(np.vdot(as_np(x[0]), as_np(x[0])) + np.vdot(as_np(x[1]), as_np(x[1])))

print("inner:", inner_xy)
print("manual inner:", manual_inner)
print("norm:", norm_x)
print("manual norm:", manual_norm)

assert np.allclose(inner_xy, manual_inner)
assert np.allclose(norm_x, manual_norm)

## 11. Context conversion and element conversion

A `TreeSpace` has one resolved context. Leaves are converted to that context. `convert_element` converts each leaf and rebuilds the same structure.


In [ ]:
ctx32 = sc.Context(sc.NumpyOps(), dtype=np.float32, check_level="standard")
T32 = T.convert(ctx32)
x32 = T.convert_element(x, ctx32)

print("space dtype before:", T.dtype)
print("space dtype after:", T32.dtype)
print("leaf dtypes after:", [leaf.dtype for leaf in T32.flatten_tree(x32)])

assert all(leaf.dtype == np.dtype("float32") for leaf in T32.flatten_tree(x32))

## 12. Leafwise Jordan/spectral functionality

A `TreeSpace` advertises Jordan/spectral functionality only when all leaves support it. For mixed spectral leaves, `spectrum` concatenates leaf spectra in deterministic leaf order.


In [ ]:
complex_ctx = sc.Context(sc.NumpyOps(), dtype=np.complex128, check_level="standard")

V = sc.ElementwiseJordanSpace((2,), complex_ctx)
H = sc.HermitianSpace(2, ctx=complex_ctx)
SpectralTree = sc.TreeSpace.from_leaf_spaces((V, H), complex_ctx)

spectral_x = (
    complex_ctx.asarray([10.0 + 0.0j, 20.0 + 0.0j]),
    complex_ctx.asarray([[1.0 + 0.0j, 0.25 + 0.0j], [0.25 + 0.0j, -2.0 + 0.0j]]),
)

spec = SpectralTree.spectrum(spectral_x)
decomp = SpectralTree.spectral_decompose(spectral_x)
rebuilt = SpectralTree.from_spectrum(decomp)

print("spectrum:", spec)
print("decomposition type:", type(decomp).__name__)
print("number of eigval leaves:", len(decomp.eigvals))

assert isinstance(decomp, sc.TreeSpectralDecomposition)
assert np.allclose(as_np(rebuilt[0]), as_np(spectral_x[0]))
assert np.allclose(as_np(rebuilt[1]), as_np(spectral_x[1]))

## 13. Tree linear operators: block diagonal

`BlockDiagonalLinOp` maps one tree to another tree leafwise.

If `A_i : X_i -> Y_i`, then

\[
A(x_1, \ldots, x_k) = (A_1 x_1, \ldots, A_k x_k).
\]


In [ ]:
A0 = sc.DenseLinOp(
    ctx.asarray([[1.0, 2.0], [3.0, 4.0]]),
    X0,
    X0,
    ctx,
)
A1 = sc.DenseLinOp(
    ctx.asarray([[1.0, 0.0, -1.0]]),
    X1,
    sc.DenseCoordinateSpace((1,), ctx),
    ctx,
)

B = sc.BlockDiagonalLinOp.from_operators((A0, A1))
Bx = B.apply(x)

print("domain:", B.dom)
print("codomain:", B.cod)
print("B(x):", Bx)

assert np.allclose(as_np(Bx[0]), [5.0, 11.0])
assert np.allclose(as_np(Bx[1]), [-20.0])

## 14. Tree linear operators: stacked operators

`StackedLinOp` maps one input into a tree of outputs.

If `A_i : X -> Y_i`, then

\[
A x = (A_1 x, \ldots, A_k x).
\]


In [ ]:
X = sc.DenseCoordinateSpace((2,), ctx)
Y0 = sc.DenseCoordinateSpace((1,), ctx)
Y1 = sc.DenseCoordinateSpace((2,), ctx)

S0 = sc.DenseLinOp(ctx.asarray([[1.0, 2.0]]), X, Y0, ctx)
S1 = sc.DenseLinOp(ctx.asarray([[3.0, 4.0], [5.0, 6.0]]), X, Y1, ctx)

S = sc.StackedLinOp.from_operators((S0, S1))
x_single = ctx.asarray([10.0, 20.0])
Sx = S.apply(x_single)

print("S(x):", Sx)
assert np.allclose(as_np(Sx[0]), [50.0])
assert np.allclose(as_np(Sx[1]), [110.0, 170.0])

## 15. Tree linear operators: sum-to-single

`SumToSingleLinOp` maps a tree input into one output by summing component images.

If `A_i : X_i -> Y`, then

\[
A(x_1, \ldots, x_k) = \sum_i A_i x_i.
\]


In [ ]:
Y = sc.DenseCoordinateSpace((2,), ctx)
C0 = sc.DenseLinOp(ctx.asarray([[1.0, 0.0], [0.0, 1.0]]), X0, Y, ctx)
C1 = sc.DenseLinOp(ctx.asarray([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]), X1, Y, ctx)

C = sc.SumToSingleLinOp.from_operators((C0, C1))
Cx = C.apply(x)

print("C(x):", Cx)
assert np.allclose(as_np(Cx), [11.0, 22.0])

## 16. Practical migration notes from ProductSpace

Use `TreeSpace` everywhere.

Old product-style intent:

```python
# old conceptual style
# P = ProductSpace((X0, X1))
# x = (x0, x1)
```

New tuple-style direct product:

```python
P = sc.TreeSpace.from_leaf_spaces((X0, X1), ctx)
x = (x0, x1)
```

New named/nested direct product:

```python
P = sc.TreeSpace.from_template({"primal": 0, "dual": 0}, (X_primal, X_dual), ctx=ctx)
x = {"primal": x_primal, "dual": x_dual}
```

Rule of thumb:

- Use `from_leaf_spaces` for tuple-structured products.
- Use `from_template` when names or nesting carry mathematical meaning.
- Use `leaf_paths` to verify ordering before constructing leafwise operators.
- Use `flatten`/`unflatten` only for dense coordinate algorithms.
- Use `flatten_tree`/`unflatten_tree` when you want to inspect or rebuild Python leaves.


## 17. Checklist for algorithm authors

When implementing an algorithm over a `TreeSpace`:

1. Decide the mathematical leaves first.
2. Pick a Python tree shape that mirrors the algorithm variables.
3. Build leaf spaces in the exact order shown by the template.
4. Inspect `leaf_paths` once.
5. Use `check` while developing.
6. Use `flatten` only at solver boundaries that need dense coordinate vectors.
7. Prefer leafwise `TreeLinOp` classes for structured operators.
8. Avoid using `TreeSpace` for tensor products; build a dedicated tensor-product space or operator instead.
